# 🔍 Demo 02 — Retrieval Hybride FinRAG

Comparaison dense vs sparse vs hybrid retrieval avec métriques.

**Objectifs :**
- Comparer Dense, BM25, Hybrid (RRF)
- Analyser l'impact des filtres temporels
- Visualiser les scores de pertinence
- Benchmarker les temps de réponse

In [ ]:
import sys
import time
from pathlib import Path

ROOT_DIR = Path('.').resolve().parent
sys.path.insert(0, str(ROOT_DIR))

from src.retrieval.vector_store import FinancialVectorStore
from src.retrieval.retriever import HybridFinancialRetriever, reciprocal_rank_fusion
from src.retrieval.reranker import CrossEncoderReRanker

vs = FinancialVectorStore()
stats = vs.get_stats()
print(f'Vector store: {stats["total_chunks"]} chunks, {stats["total_sources"]} sources')

## 1. Dense Retrieval

In [ ]:
query = "chiffre d'affaires Apple Services FY2023"

start = time.time()
dense_results = vs.similarity_search(query, k=5)
dense_time = time.time() - start

print(f'Dense retrieval: {len(dense_results)} résultats en {dense_time*1000:.0f}ms')
print()
for i, (doc, score) in enumerate(dense_results):
    print(f'{i+1}. Score: {score:.3f} | {doc.metadata.get("filename","?")} p.{doc.metadata.get("page_number","?")}')
    print(f'   {doc.page_content[:100]}...')

## 2. Sparse Retrieval (BM25)

In [ ]:
from src.retrieval.retriever import BM25Retriever

# Get all docs for BM25
all_docs = vs.get_all_documents()
if all_docs:
    start = time.time()
    bm25 = BM25Retriever(all_docs)
    bm25_results = bm25.search(query, k=5)
    bm25_time = time.time() - start
    
    print(f'BM25 retrieval: {len(bm25_results)} résultats en {bm25_time*1000:.0f}ms')
    for i, (doc, score) in enumerate(bm25_results):
        print(f'{i+1}. Score: {score:.3f} | {doc.metadata.get("filename","?")} p.{doc.metadata.get("page_number","?")}')
        print(f'   {doc.page_content[:100]}...')
else:
    print('Vector store vide. Indexez des documents d\'abord.')

## 3. Hybrid Retrieval (RRF)

In [ ]:
retriever = HybridFinancialRetriever(vector_store=vs, top_k=5)

start = time.time()
hybrid_results = retriever.retrieve(query, use_hybrid=True)
hybrid_time = time.time() - start

print(f'Hybrid RRF: {len(hybrid_results)} résultats en {hybrid_time*1000:.0f}ms')
for i, (doc, score) in enumerate(hybrid_results):
    print(f'{i+1}. Score RRF: {score:.4f} | {doc.metadata.get("filename","?")} p.{doc.metadata.get("page_number","?")}')
    print(f'   {doc.page_content[:100]}...')

## 4. Comparaison Visuelle

In [ ]:
import plotly.graph_objects as go

# Normalize scores for comparison
def get_source_scores(results, label):
    return [{
        'method': label,
        'source': doc.metadata.get('filename', '?')[:20],
        'score': score,
        'rank': i + 1
    } for i, (doc, score) in enumerate(results)]

if dense_results and hybrid_results:
    fig = go.Figure()
    
    methods = [
        (dense_results, 'Dense', '#3B82F6'),
        (hybrid_results, 'Hybrid RRF', '#F0B429'),
    ]
    
    for results, name, color in methods:
        scores = [s for _, s in results[:5]]
        labels = [f"{doc.metadata.get('filename','?')[:15]}" for doc, _ in results[:5]]
        fig.add_trace(go.Bar(
            name=name, x=labels, y=scores, marker_color=color
        ))
    
    fig.update_layout(
        title='Comparaison Dense vs Hybrid RRF',
        template='plotly_dark',
        barmode='group',
        xaxis_title='Source',
        yaxis_title='Score de pertinence'
    )
    fig.show()
else:
    print('Résultats insuffisants pour la comparaison')

## 5. Re-ranking CrossEncoder

In [ ]:
reranker = CrossEncoderReRanker(top_k=3)

if hybrid_results:
    start = time.time()
    reranked = reranker.rerank(query, hybrid_results)
    rerank_time = time.time() - start
    
    print(f'Re-ranking: {len(reranked)} résultats en {rerank_time*1000:.0f}ms')
    print(f'Cache stats: {reranker.get_cache_stats()}')
    print()
    for i, (doc, score) in enumerate(reranked):
        print(f'{i+1}. CrossEncoder score: {score:.4f}')
        print(f'   {doc.metadata.get("filename","?")} | {doc.page_content[:80]}...')
else:
    print('Pas de résultats hybrides à re-ranker')